# Data Prep

In [1]:
import pandas as pd
import numpy as np
import gc

## Prep endurance data

In [2]:
endurance_race_1_path = "barber-motorsports-park/barber/23_AnalysisEnduranceWithSections_Race 1_Anonymized.CSV"
endurance_race_2_path = "barber-motorsports-park/barber/23_AnalysisEnduranceWithSections_Race 2_Anonymized.CSV"

In [3]:
endurance_race_1 = pd.read_csv(endurance_race_1_path, sep =';')
endurance_race_2 = pd.read_csv(endurance_race_2_path, sep =';')

endurance_race_1['meta_session'] = 'R1'
endurance_race_2['meta_session'] = 'R2'

endurance = pd.concat([endurance_race_1, endurance_race_2])


In [4]:
def prepare_endurance(df):
    df.columns=df.columns.str.strip()
    df.columns = df.columns.str.lower()
    df.rename(columns={'number': 'vehicle_number',"lap_number":"lap"}, inplace=True)
    df.dropna(axis=1, inplace=True)
    return df

In [38]:
endurance = prepare_endurance(endurance)
endurance.to_csv("simulation-data/endurance.csv")

In [5]:
# Prep telemetry data

telemetry_r1_path = "barber-motorsports-park/barber/R1_barber_telemetry_data.CSV"
telemetry_r2_path = "barber-motorsports-park/barber/R2_barber_telemetry_data.CSV"

telemetry_r1 = pd.read_csv(telemetry_r1_path)
telemetry_r1_parquet_path = "barber-motorsports-park/barber/R1_barber_telemetry_data.parquet"
telemetry_r1.to_parquet(telemetry_r1_parquet_path, engine="pyarrow", index=False, compression="zstd", compression_level=12)

del telemetry_r1
gc.collect()

telemetry_r2 = pd.read_csv(telemetry_r2_path)
telemetry_r2_parquet_path = "barber-motorsports-park/barber/R2_barber_telemetry_data.parquet"
telemetry_r2.to_parquet(telemetry_r2_parquet_path, engine="pyarrow", index=False, compression="zstd", compression_level=12)


del telemetry_r2
gc.collect()



0

In [4]:
# Telemetry

telemetry_r1_parquet_path = "barber-motorsports-park/barber/R1_barber_telemetry_data.parquet"
telemetry_r2_parquet_path = "barber-motorsports-park/barber/R2_barber_telemetry_data.parquet"

tele_R1 = pd.read_parquet(telemetry_r1_parquet_path)
tele_R2 = pd.read_parquet(telemetry_r2_parquet_path)

df_tel_long = pd.concat([tele_R1, tele_R2])
df_tel_long.dropna(axis=1, inplace=True)



In [17]:
df_tel_long.drop(columns=['meta_event', "meta_source","original_vehicle_id","vehicle_id","outing","timestamp"],inplace=True)

In [18]:
df_tel_long.to_parquet("simulation-data/telemetry.parquet")

In [5]:
# Tele wide 
import pandas as pd

def reshape_telemetry(
    df: pd.DataFrame,
    index_cols=("meta_session", "vehicle_number", "meta_time"),
    name_col="telemetry_name",
    value_col="telemetry_value",
) -> pd.DataFrame:
    """
    Convert long-format telemetry into wide format.

    Expected input columns (default):
      - 'meta_session'
      - 'vehicle_number'
      - 'meta_time'      (ISO string; will be parsed to datetime)
      - 'telemetry_name' (e.g. 'accx_can', 'VBOX_Lat_Min', 'speed', ...)
      - 'telemetry_value'

    Returns a DataFrame with one row per (meta_session, vehicle_number, meta_time)
    and one column per telemetry_name + the index columns.
    """

    df = df.copy()

    # Parse time if it's still a string
    if "meta_time" in df.columns and df["meta_time"].dtype == object:
        df["meta_time"] = pd.to_datetime(df["meta_time"], utc=True, errors="coerce")

    # Pivot to wide
    wide = (
        df
        .pivot_table(
            index=list(index_cols),
            columns=name_col,
            values=value_col,
            aggfunc="first"   # assume the combination is unique
        )
        .reset_index()
    )

    # remove the column index name created by pivot_table
    wide.columns.name = None

    # Optional: reorder columns if you want a specific order
    desired_order = [
        "meta_session",
        "vehicle_number",
        "meta_time",
        "Laptrigger_lapdist_dls",
        "Steering_Angle",
        "VBOX_Lat_Min",
        "VBOX_Long_Minutes",
        "accx_can",
        "accy_can",
        "aps",
        "gear",
        "nmot",
        "pbrake_f",
        "pbrake_r",
        "speed",
    ]
    cols_present = [c for c in desired_order if c in wide.columns]
    other_cols = [c for c in wide.columns if c not in cols_present]

    wide = wide[cols_present + other_cols]

    return wide


In [6]:
tele_wide = reshape_telemetry(df_tel_long)

In [ ]:
tele_wide.to_parquet('simulation-data/telemetry_wide.parquet', index=False)

# END

In [ ]:
# BOUNDS

# bounds1 = pd.read_csv('bounds1.csv')
# bounds2 = pd.read_csv('bounds2.csv')
# bounds = pd.concat([bounds1,bounds2])
# bounds.to_csv("simulation-data/bounds.csv")